In [4]:
import sys, pandas as pd
print("Python:", sys.version)
print("Pandas:", pd.__version__)

Python: 3.11.7 (main, Jan  8 2024, 20:44:44) [GCC 9.4.0]
Pandas: 2.3.2


In [1]:
import os, glob
import pandas as pd

csv_path = os.getenv("DATA_CSV") or sorted(glob.glob("../data/*.csv"), key=os.path.getsize, reverse=True)[0]
df = pd.read_csv(csv_path, low_memory=False)
print("Loaded:", csv_path, "rows:", len(df), "cols:", df.shape[1])
df.head()

Loaded: ../data/used_car_sales.csv rows: 122144 cols: 13


,ID,pricesold,yearsold,zipcode,Mileage,Make,Model,Year,Trim,Engine,BodyType,NumCylinders,DriveType
0,137178,7500,2020,786**,84430,Ford,Mustang,1988,LX,5.0L Gas V8,Sedan,0,RWD
1,96705,15000,2019,81006,0,Replica/Kit Makes,Jaguar Beck Lister,1958,NaN,383 Fuel injected,Convertible,8,RWD
2,119660,8750,2020,33449,55000,Jaguar,XJS,1995,2+2 Cabriolet,4.0L In-Line 6 Cylinder,Convertible,6,RWD
3,80773,11600,2019,07852,97200,Ford,Mustang,1968,Stock,289 cu. in. V8,Coupe,8,RWD
4,64287,44000,2019,07728,40703,Porsche,911,2002,Turbo X-50,3.6L,Coupe,6,AWD


In [2]:
import re
def find_col(cands, cols):
    rx = re.compile("|".join([fr"\b{re.escape(c)}\b" for c in cands]), re.I)
    hits = [c for c in cols if rx.search(c)]
    return hits[0] if hits else None

cols = df.columns.tolist()
price   = find_col(["price","list_price","selling_price"], cols)
year    = find_col(["year","model_year"], cols)
miles   = find_col(["mileage","odometer"], cols)
make    = find_col(["make","brand"], cols)
model   = find_col(["model"], cols)
state   = find_col(["state","region"], cols)
listdate= find_col(["listed_at","posting_date","date","created_at"], cols)

print("Detected:", dict(price=price, year=year, miles=miles, make=make, model=model, state=state, listdate=listdate))
print("\nMissing % (top 15):")
print((df.isna().mean().sort_values(ascending=False).head(15)*100).round(1))

if price and year and miles and make and model:
    print("\nPrice summary:")
    print(df[price].describe(percentiles=[.01,.05,.5,.95,.99]).to_string())
    print("\nMileage summary:")
    print(df[miles].describe(percentiles=[.01,.05,.5,.95,.99]).to_string())
    print("\nTop makes:")
    print(df[make].value_counts().head(15))
else:
    print("\n❗ One of the key columns is missing—note the detections above.")


Detected: {'price': None, 'year': 'Year', 'miles': 'Mileage', 'make': 'Make', 'model': 'Model', 'state': None, 'listdate': None}

Missing % (top 15):
Trim            40.1
Engine          22.3
DriveType       20.4
BodyType        17.0
zipcode          0.7
Model            0.5
yearsold         0.0
pricesold        0.0
ID               0.0
Mileage          0.0
Make             0.0
Year             0.0
NumCylinders     0.0
dtype: float64

❗ One of the key columns is missing—note the detections above.


In [3]:
sample = df.sample(min(20000, len(df)), random_state=42)
sample.to_csv("../data/sample.csv", index=False)
len(sample)


20000